# 02 — Baseline CNN

This notebook demonstrates the simple CNN baseline used to verify the end-to-end MRI classification pipeline.

The baseline is intentionally lightweight. Its purpose is to provide a reference implementation for data loading, weighted training, checkpointing, prediction aggregation, and participant-level evaluation.

Development experiments shown through this pipeline are not treated as final scientific results.


## 1. Project Setup

The notebook imports the reusable implementation from the `src/` directory.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MRI_XAI_Robustness"
)

SRC_DIR = PROJECT_ROOT / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Project root:', PROJECT_ROOT)


## 2. Reusable Project Modules

The baseline notebook does not redefine the dataset, training loop, evaluation logic, or metrics.


In [ ]:
from dataset import NACC25DDataset
from models.baseline import BaselineCNN
from train import train_binary_model
from evaluate import collect_participant_predictions
from metrics import calculate_participant_metrics
from utils import (
    DEFAULT_SEED,
    set_random_seed,
    get_device,
    count_parameters,
)

set_random_seed(DEFAULT_SEED)
device = get_device()

print('Random seed:', DEFAULT_SEED)
print('Device:', device)


## 3. Baseline Architecture

The baseline consists of three convolutional blocks followed by adaptive average pooling and a single binary classification output.

The model contains 23,649 parameters.


In [ ]:
baseline_model = BaselineCNN().to(device)

parameter_counts = count_parameters(
    baseline_model
)

print(baseline_model)

print(
    'Total parameters:',
    f"{parameter_counts['total_parameters']:,}"
)

print(
    'Trainable parameters:',
    f"{parameter_counts['trainable_parameters']:,}"
)

assert parameter_counts['total_parameters'] == 23649

print('PASS: Baseline CNN initialized correctly.')


## 4. Development Data

The example training workflow uses a small development subset drawn only from the frozen NACC training partition.

This subset is used to verify that the modeling pipeline functions correctly. It is not used to report final scientific performance.

Controlled participant manifests and MRI stacks are not included in the repository.


In [ ]:
MANIFEST_DIR = PROJECT_ROOT / 'data' / 'manifests'

development_manifest_path = (
    MANIFEST_DIR
    / 'nacc_pipeline_development_50.csv'
)

development_stack_root = (
    PROJECT_ROOT
    / 'data'
    / 'pipeline_development_50'
    / '25d_stacks_normalized'
)

data_available = (
    development_manifest_path.exists()
    and development_stack_root.exists()
)

print('Development data available:', data_available)

if data_available:

    development_df = pd.read_csv(
        development_manifest_path
    )

    print('Participants:', len(development_df))
    print(development_df['label'].value_counts())

else:

    print(
        'Controlled development data are not available '
        'in this environment.'
    )


## 5. Weighted Binary Training

The natural NACC class distribution is retained.

Class imbalance is handled using `BCEWithLogitsLoss` with a positive-class weight calculated from the full frozen NACC training partition:

`2394 / 365 ≈ 6.5589`

The weight is not recalculated from the small development subset.


In [ ]:
NACC_TRAIN_NORMAL = 2394
NACC_TRAIN_AD = 365

positive_class_weight = (
    NACC_TRAIN_NORMAL
    / NACC_TRAIN_AD
)

pos_weight_tensor = torch.tensor(
    [positive_class_weight],
    dtype=torch.float32,
    device=device,
)

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)

optimizer = torch.optim.Adam(
    baseline_model.parameters(),
    lr=1e-3,
)

print(
    'Positive-class weight:',
    f'{positive_class_weight:.4f}'
)

print('Loss:', criterion)
print('Optimizer: Adam, learning rate = 1e-3')


## 6. Training Workflow

The reusable `train_binary_model` function handles:

- weighted binary training
- validation-loss calculation
- checkpointing
- CSV logging
- early stopping

Training and validation datasets must be created at the participant level so that all five stacks from a participant remain in the same partition.

The code below shows the expected training call after `train_loader` and `validation_loader` have been created.


In [ ]:
# Example training call
#
# history = train_binary_model(
#     model=baseline_model,
#     train_loader=train_loader,
#     validation_loader=validation_loader,
#     criterion=criterion,
#     optimizer=optimizer,
#     device=device,
#     checkpoint_path=(
#         PROJECT_ROOT
#         / 'checkpoints'
#         / 'baseline_best.pt'
#     ),
#     log_path=(
#         PROJECT_ROOT
#         / 'results'
#         / 'logs'
#         / 'baseline_training_log.csv'
#     ),
#     max_epochs=20,
#     patience=5,
# )

print(
    'Training function imported and ready: '
    'train_binary_model'
)


## 7. Participant-Level Evaluation

Each participant contributes five 2.5D stacks.

The model first produces a probability for each stack. The five probabilities are then averaged to obtain one participant-level dementia probability.

Primary evaluation is therefore performed at the participant level rather than treating individual stacks as independent participants.


In [ ]:
# Example participant-level prediction workflow
#
# stack_predictions, participant_predictions = (
#     collect_participant_predictions(
#         model=baseline_model,
#         data_loader=validation_loader,
#         device=device,
#     )
# )
#
# metrics = calculate_participant_metrics(
#     participant_predictions,
#     threshold=0.5,
# )
#
# print(metrics)

print(
    'Participant-level evaluation functions are ready.'
)


## 8. Classification Threshold

A threshold of 0.5 may be used during pipeline-development checks, but it is not automatically treated as the final study threshold.

For the final experiment, the classification threshold will be selected using the official NACC validation partition and then frozen before evaluation on:

- the official NACC test partition, and
- the ADNI external-validation cohort.

ADNI is not used for threshold selection.


## 9. Summary

This notebook demonstrates the baseline modeling workflow without embedding controlled MRI data or presenting development checks as final scientific results.

Reusable implementation:

- `src/models/baseline.py`
- `src/dataset.py`
- `src/train.py`
- `src/evaluate.py`
- `src/metrics.py`
- `src/utils.py`

The baseline serves as a simple reference model for the main ConvNeXt V2 Tiny pipeline.
